In [43]:
%useLatestDescriptors
%use dataframe(1.0.0-Beta5n)
%use kandy
%use serialization

In [44]:
@file:Repository("https://repo.gradle.org/gradle/libs-releases") //
@file:DependsOn("org.gradle:gradle-tooling-api:9.6.0")

import org.gradle.tooling.GradleConnector
import java.io.File

GradleConnector.newConnector().forProjectDirectory(File("./../../")).connect().use { connection ->
    connection.newBuild()
            .forTasks("clean", "reverseBytesBenchmark", "reverseBitsBenchmark")
            .setStandardOutput(System.out)
            .setStandardError(System.err)
            .run()
}

Calculating task graph as no cached configuration is available for tasks: clean reverseBytesBenchmark reverseBitsBenchmark
Type-safe project accessors is an incubating feature.
> Task :karbide-benchmarks:cleanMacosArm64Test UP-TO-DATE
> Task :karbide-benchmarks:cleanWasmWasiNodeTest UP-TO-DATE
> Task :karbide-core:cleanJsBrowserTest UP-TO-DATE
> Task :karbide-core:cleanWasmJsBrowserTest UP-TO-DATE
> Task :kotlinWasmKotlinNpmCachesSetup
> Task :karbide-core:cleanWasmJsNodeTest UP-TO-DATE
> Task :karbide-benchmarks:cleanJsNodeTest UP-TO-DATE
> Task :karbide-core:cleanLinuxX64Test UP-TO-DATE
> Task :karbide-benchmarks:cleanMingwX64Test UP-TO-DATE
> Task :karbide-benchmarks:cleanIosSimulatorArm64Test UP-TO-DATE
> Task :karbide-core:cleanWasmWasiNodeTest UP-TO-DATE
> Task :karbide-benchmarks:kmpPartiallyResolvedDependenciesChecker
> Task :karbide-core:cleanMingwX64Test UP-TO-DATE
> Task :karbide-benchmarks:cleanWasmJsNodeTest UP-TO-DATE
> Task :kotlinKotlinNpmCachesSetup
> Task :karbide-cor

In [51]:
@file:OptIn(ExperimentalSerializationApi::class)

import kotlinx.serialization.ExperimentalSerializationApi
import kotlinx.serialization.SerialName
import kotlinx.serialization.Serializable
import kotlinx.serialization.json.Json
import kotlinx.serialization.json.decodeFromStream
import org.jetbrains.kotlinx.kandy.dsl.plot
import org.jetbrains.kotlinx.kandy.letsplot.layers.bars
import java.nio.file.Path
import kotlin.io.path.PathWalkOption
import kotlin.io.path.extension
import kotlin.io.path.inputStream
import kotlin.io.path.name
import kotlin.io.path.nameWithoutExtension
import kotlin.io.path.walk

fun String.suffixSimilarity(other: String): Double {
    val a = this.reversed()
    val b = other.reversed()
    val maxLen = maxOf(a.length, b.length)
    if (maxLen == 0) return 1.0
    var match = 0
    for (i in 0 until minOf(a.length, b.length)) {
        if (a[i] == b[i]) match++ else break
    }
    return match.toDouble() / maxLen
}

inline fun <T> List<T>.groupBySimilarityChain(crossinline selector: (T) -> String): List<T> {
    if (isEmpty()) return emptyList()
    val unused = toMutableSet()
    val result = ArrayList<T>()
    var current = unused.first()
    result += current
    unused -= current
    while (unused.isNotEmpty()) {
        val next = unused.maxBy { selector(it).suffixSimilarity(selector(current)) }
        result += next
        unused -= next
        current = next
    }
    return result
}

@Serializable
data class PrimaryMetric(
    val score: Double,
    val scoreError: Double,
    val scoreUnit: String
)

@Serializable
data class Benchmark(
    @SerialName("benchmark") val name: String,
    val warmupIterations: Int,
    val measurementIterations: Int,
    val primaryMetric: PrimaryMetric
) {
    inline val cleanName: String
        get() = name.substringBeforeLast('.').substringAfterLast('.')
}

val json: Json = Json { ignoreUnknownKeys = true }

Path.of("./../build/reports/benchmarks")
        .walk()
        .filter { filePath -> filePath.extension == "json" }
        .map { filePath ->
            filePath.inputStream().use { stream ->
                filePath to json.decodeFromStream<List<Benchmark>>(stream)
            }
        }.map { (filePath, report) ->
            val sortedReport = report.groupBySimilarityChain(Benchmark::name)
            val plotName = filePath.parent.parent.name
            val platform = filePath.nameWithoutExtension
            val plot = plot {
                layout {
                    size = 1200 to 800
                    title = "$plotName ($platform)" // Parent of parent is suite name
                    xAxisLabel = "Implementation"
                    yAxisLabel = "ops/s"
                }
                val scores = sortedReport.map { benchmark -> benchmark.primaryMetric.score }
                bars {
                    x(sortedReport.map(Benchmark::cleanName))
                    y(scores) {
                        axis {
                            breaks(format = "{,.0f}")
                        }
                    }
                }
                errorBars {
                    x(sortedReport.map(Benchmark::cleanName))
                    val errors = sortedReport.map { benchmark -> benchmark.primaryMetric.scoreError }
                    yMin(scores.zip(errors).map { (score, error) -> score - error })
                    yMax(scores.zip(errors).map { (score, error) -> score + error })
                }
            }
            plot.save("./../../../docs/${plotName}_$platform.png")
            plot
        }.forEach(::DISPLAY)

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="FLB3Io" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("FLB3Io");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"reverseBits (jvm)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"ops/s"
}
},
"data":{
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":"{,.0f}",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["CommonByteReverseBitsBenchmark","KarbideByteReverseBitsBenchmark","CommonIntReverseBitsBenchmark","KarbideIntReverseBitsBenchmark","CommonShortReverseBitsBenchmark","KarbideShortReverseBitsBenchmark","CommonLongReverseBitsBenchmark","KarbideLongReverseBitsBenchmark"],
"y":[9.072482145927645E7,1.1481926821922156E8,4.93574425125402E7,2.2763286380904895E8,7.909769051264413E7,1.342438472777873E8,2.527449562101953E7,1.6930634030616394E8]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"ymin":"ymin",
"ymax":"ymax"
},
"stat":"identity",
"data":{
"ymin":[8.670204197548413E7,1.0964663360437132E8,4.6995469424580686E7,2.1989135690305603E8,7.681370724175183E7,1.3121931071317458E8,2.4808430326299634E7,1.6540730228524667E8],
"ymax":[9.474760094306877E7,1.199919028340718E8,5.171941560049971E7,2.3537437071504188E8,8.138167378353642E7,1.372683838424E8,2.574056091573943E7,1.732053783270812E8],
"x":["CommonByteReverseBitsBenchmark","KarbideByteReverseBitsBenchmark","CommonIntReverseBitsBenchmark","KarbideIntReverseBitsBenchmark","CommonShortReverseBitsBenchmark","KarbideShortReverseBitsBenchmark","CommonLongReverseBitsBenchmark","KarbideLongReverseBitsBenchmark"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"ymin"
},{
"type":"float",
"column":"ymax"
}]
}
}],
"spec_id":"415"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="2ttqbH" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("2ttqbH");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"reverseBits (js)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"ops/s"
}
},
"data":{
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":"{,.0f}",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["CommonByteReverseBitsBenchmark","KarbideByteReverseBitsBenchmark","CommonIntReverseBitsBenchmark","KarbideIntReverseBitsBenchmark","CommonShortReverseBitsBenchmark","KarbideShortReverseBitsBenchmark","CommonLongReverseBitsBenchmark","KarbideLongReverseBitsBenchmark"],
"y":[4.078568895547937E7,6.120240139901028E7,2.8713975378522784E7,8.660739689288872E7,3.50101261788706E7,6.498062660473057E7,214931.08929726708,2945304.39610268]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"ymin":"ymin",
"ymax":"ymax"
},
"stat":"identity",
"data":{
"ymin":[3.850183859400601E7,5.725347908225498E7,2.7669120752002593E7,8.056797935837565E7,3.3451397026793092E7,5.91956789330637E7,206718.04348123347,2833379.7258540667],
"ymax":[4.306953931695273E7,6.515132371576557E7,2.9758830005042974E7,9.26468144274018E7,3.6568855330948114E7,7.076557427639744E7,223144.1351133007,3057229.066351293],
"x":["CommonByteReverseBitsBenchmark","KarbideByteReverseBitsBenchmark","CommonIntReverseBitsBenchmark","KarbideIntReverseBitsBenchmark","CommonShortReverseBitsBenchmark","KarbideShortReverseBitsBenchmark","CommonLongReverseBitsBenchmark","KarbideLongReverseBitsBenchmark"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"ymin"
},{
"type":"float",
"column":"ymax"
}]
}
}],
"spec_id":"419"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="9a137J" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("9a137J");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"reverseBits (wasmWasi)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"ops/s"
}
},
"data":{
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":"{,.0f}",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["CommonByteReverseBitsBenchmark","KarbideByteReverseBitsBenchmark","CommonIntReverseBitsBenchmark","KarbideIntReverseBitsBenchmark","CommonShortReverseBitsBenchmark","KarbideShortReverseBitsBenchmark","CommonLongReverseBitsBenchmark","KarbideLongReverseBitsBenchmark"],
"y":[3.732894527402465E7,3.159268854855009E7,2.3856978617506467E7,2.457299011507195E7,2.7536809318665057E7,3.024753290142966E7,1.5766944545019388E7,1.6138311064926201E7]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"ymin":"ymin",
"ymax":"ymax"
},
"stat":"identity",
"data":{
"ymin":[3.61715712752553E7,3.0568224233650383E7,2.183333238182288E7,2.418702203756285E7,2.6499492055121243E7,3.000755771378854E7,1.552508263126736E7,1.5708397809512941E7],
"ymax":[3.8486319272794E7,3.26171528634498E7,2.5880624853190053E7,2.4958958192581046E7,2.857412658220887E7,3.0487508089070782E7,1.6008806458771417E7,1.6568224320339462E7],
"x":["CommonByteReverseBitsBenchmark","KarbideByteReverseBitsBenchmark","CommonIntReverseBitsBenchmark","KarbideIntReverseBitsBenchmark","CommonShortReverseBitsBenchmark","KarbideShortReverseBitsBenchmark","CommonLongReverseBitsBenchmark","KarbideLongReverseBitsBenchmark"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"ymin"
},{
"type":"float",
"column":"ymax"
}]
}
}],
"spec_id":"423"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="JvUCpb" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("JvUCpb");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"reverseBits (wasmJs)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"ops/s"
}
},
"data":{
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":"{,.0f}",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["CommonByteReverseBitsBenchmark","KarbideByteReverseBitsBenchmark","CommonIntReverseBitsBenchmark","KarbideIntReverseBitsBenchmark","CommonShortReverseBitsBenchmark","KarbideShortReverseBitsBenchmark","CommonLongReverseBitsBenchmark","KarbideLongReverseBitsBenchmark"],
"y":[3.407783610195844E7,3.458050080341482E7,2.5407683569705762E7,2.4585768260399323E7,2.680709148163414E7,2.6128885434638593E7,1.4872141136330027E7,1.6148482646097202E7]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"ymin":"ymin",
"ymax":"ymax"
},
"stat":"identity",
"data":{
"ymin":[3.2749944786913026E7,3.354556793498584E7,2.499569216866706E7,2.401833738718912E7,2.6224842275300063E7,2.5674302194048014E7,1.4523751896796105E7,1.5762107509167766E7],
"ymax":[3.540572741700385E7,3.5615433671843804E7,2.5819674970744465E7,2.5153199133609526E7,2.7389340687968217E7,2.6583468675229173E7,1.5220530375863949E7,1.6534857783026638E7],
"x":["CommonByteReverseBitsBenchmark","KarbideByteReverseBitsBenchmark","CommonIntReverseBitsBenchmark","KarbideIntReverseBitsBenchmark","CommonShortReverseBitsBenchmark","KarbideShortReverseBitsBenchmark","CommonLongReverseBitsBenchmark","KarbideLongReverseBitsBenchmark"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"ymin"
},{
"type":"float",
"column":"ymax"
}]
}
}],
"spec_id":"427"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="nhgmzO" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("nhgmzO");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"reverseBits (linuxX64)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"ops/s"
}
},
"data":{
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":"{,.0f}",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["CommonByteReverseBitsBenchmark","KarbideByteReverseBitsBenchmark","CommonIntReverseBitsBenchmark","KarbideIntReverseBitsBenchmark","CommonShortReverseBitsBenchmark","KarbideShortReverseBitsBenchmark","CommonLongReverseBitsBenchmark","KarbideLongReverseBitsBenchmark"],
"y":[6.955313735442151E7,7.100816271399361E7,2.78262930448846E7,4.575895891859562E7,3.855098249185678E7,4.106725876902791E7,1.2399975398557404E7,3.3604578744471595E7]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"ymin":"ymin",
"ymax":"ymax"
},
"stat":"identity",
"data":{
"ymin":[6.744801812046418E7,7.016589958573255E7,2.6481914129402738E7,4.536956075958409E7,3.469688836311707E7,3.894362259763993E7,1.1643543635290688E7,3.3039637914319023E7],
"ymax":[7.165825658837885E7,7.185042584225467E7,2.917067196036646E7,4.614835707760715E7,4.24050766205965E7,4.319089494041589E7,1.315640716182412E7,3.4169519574624166E7],
"x":["CommonByteReverseBitsBenchmark","KarbideByteReverseBitsBenchmark","CommonIntReverseBitsBenchmark","KarbideIntReverseBitsBenchmark","CommonShortReverseBitsBenchmark","KarbideShortReverseBitsBenchmark","CommonLongReverseBitsBenchmark","KarbideLongReverseBitsBenchmark"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"ymin"
},{
"type":"float",
"column":"ymax"
}]
}
}],
"spec_id":"431"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="59Ykrx" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("59Ykrx");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"reverseBytes (js)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"ops/s"
}
},
"data":{
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":"{,.0f}",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["CommonIntReverseBytesBenchmark","KarbideIntReverseBytesBenchmark","CommonShortReverseBytesBenchmark","KarbideShortReverseBytesBenchmark","CommonLongReverseBytesBenchmark","KarbideLongReverseBytesBenchmark"],
"y":[6.836958963501246E7,8.171330675257477E7,6.062537966454531E7,6.2567584616777696E7,1305730.0270212705,3921631.8966558785]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"ymin":"ymin",
"ymax":"ymax"
},
"stat":"identity",
"data":{
"ymin":[6.197723706163994E7,7.536041381455679E7,5.791856503970153E7,5.774473696275231E7,1279981.8863042425,3809258.8209558562],
"ymax":[7.476194220838498E7,8.806619969059275E7,6.3332194289389096E7,6.739043227080308E7,1331478.1677382984,4034004.972355901],
"x":["CommonIntReverseBytesBenchmark","KarbideIntReverseBytesBenchmark","CommonShortReverseBytesBenchmark","KarbideShortReverseBytesBenchmark","CommonLongReverseBytesBenchmark","KarbideLongReverseBytesBenchmark"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"ymin"
},{
"type":"float",
"column":"ymax"
}]
}
}],
"spec_id":"435"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonIntReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 KarbideIntReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 CommonShortReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 KarbideShortReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 CommonLongReverseBytesBenchmark 
 
 

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="BUJVfR" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("BUJVfR");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"reverseBytes (wasmWasi)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"ops/s"
}
},
"data":{
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":"{,.0f}",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["CommonIntReverseBytesBenchmark","KarbideIntReverseBytesBenchmark","CommonShortReverseBytesBenchmark","KarbideShortReverseBytesBenchmark","CommonLongReverseBytesBenchmark","KarbideLongReverseBytesBenchmark"],
"y":[4.95873985129307E7,4.7425848970550895E7,4.2666242401382126E7,4.18955721687449E7,3.656857121789331E7,4.053599180969399E7]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"ymin":"ymin",
"ymax":"ymax"
},
"stat":"identity",
"data":{
"ymin":[4.730843632868351E7,4.53944461292396E7,4.221487673113609E7,4.085061814168262E7,3.554124297865679E7,3.964743566703236E7],
"ymax":[5.186636069717789E7,4.945725181186219E7,4.311760807162816E7,4.294052619580718E7,3.759589945712983E7,4.142454795235562E7],
"x":["CommonIntReverseBytesBenchmark","KarbideIntReverseBytesBenchmark","CommonShortReverseBytesBenchmark","KarbideShortReverseBytesBenchmark","CommonLongReverseBytesBenchmark","KarbideLongReverseBytesBenchmark"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"ymin"
},{
"type":"float",
"column":"ymax"
}]
}
}],
"spec_id":"439"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonIntReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 KarbideIntReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 CommonShortReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 KarbideShortReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 CommonLongReverseBytes

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="IXtfH2" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("IXtfH2");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"reverseBytes (wasmJs)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"ops/s"
}
},
"data":{
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":"{,.0f}",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["CommonIntReverseBytesBenchmark","KarbideIntReverseBytesBenchmark","CommonShortReverseBytesBenchmark","KarbideShortReverseBytesBenchmark","CommonLongReverseBytesBenchmark","KarbideLongReverseBytesBenchmark"],
"y":[5.613726170019398E7,4.6137507656912394E7,4.523587904783517E7,4.284716995585446E7,3.2643535379630297E7,3.978179150211297E7]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"ymin":"ymin",
"ymax":"ymax"
},
"stat":"identity",
"data":{
"ymin":[5.380014995892234E7,4.528163378346636E7,4.33133847098392E7,4.17477068210518E7,3.1696250241171695E7,3.9341171207277335E7],
"ymax":[5.8474373441465616E7,4.6993381530358426E7,4.715837338583114E7,4.394663309065712E7,3.35908205180889E7,4.0222411796948604E7],
"x":["CommonIntReverseBytesBenchmark","KarbideIntReverseBytesBenchmark","CommonShortReverseBytesBenchmark","KarbideShortReverseBytesBenchmark","CommonLongReverseBytesBenchmark","KarbideLongReverseBytesBenchmark"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"ymin"
},{
"type":"float",
"column":"ymax"
}]
}
}],
"spec_id":"443"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonIntReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 KarbideIntReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 CommonShortReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 KarbideShortReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 CommonLongR

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="pDExY3" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("pDExY3");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"reverseBytes (linuxX64)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"ops/s"
}
},
"data":{
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":"{,.0f}",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["CommonIntReverseBytesBenchmark","KarbideIntReverseBytesBenchmark","CommonShortReverseBytesBenchmark","KarbideShortReverseBytesBenchmark","CommonLongReverseBytesBenchmark","KarbideLongReverseBytesBenchmark"],
"y":[6.491770717579985E7,6.411039902536925E7,4.412039668756668E7,5.1194240714677826E7,4.9857079345477E7,4.969943293825213E7]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"ymin":"ymin",
"ymax":"ymax"
},
"stat":"identity",
"data":{
"ymin":[6.1147043908323616E7,5.952542541239682E7,4.125622587380884E7,5.0505093381478876E7,4.620836456398991E7,4.84494143889011E7],
"ymax":[6.868837044327608E7,6.869537263834168E7,4.698456750132453E7,5.1883388047876775E7,5.350579412696409E7,5.094945148760316E7],
"x":["CommonIntReverseBytesBenchmark","KarbideIntReverseBytesBenchmark","CommonShortReverseBytesBenchmark","KarbideShortReverseBytesBenchmark","CommonLongReverseBytesBenchmark","KarbideLongReverseBytesBenchmark"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"ymin"
},{
"type":"float",
"column":"ymax"
}]
}
}],
"spec_id":"447"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonIntReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 KarbideIntReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 CommonShortReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 KarbideShortReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 CommonLongReverseBytesBenchmark 

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="TlvojG" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("TlvojG");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"reverseBytes (jvm)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"ops/s"
}
},
"data":{
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":"{,.0f}",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["CommonIntReverseBytesBenchmark","KarbideIntReverseBytesBenchmark","CommonShortReverseBytesBenchmark","KarbideShortReverseBytesBenchmark","CommonLongReverseBytesBenchmark","KarbideLongReverseBytesBenchmark"],
"y":[2.5835043715228477E8,3.0745796601652825E8,1.7384942247598106E8,1.8849470069093934E8,1.5478865456572962E8,2.180878178958107E8]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"ymin":"ymin",
"ymax":"ymax"
},
"stat":"identity",
"data":{
"ymin":[2.5151618372028488E8,2.978922551881137E8,1.552007873749324E8,1.6310127471711755E8,1.4732904660043576E8,1.8353493274142066E8],
"ymax":[2.6518469058428466E8,3.170236768449428E8,1.924980575770297E8,2.1388812666476113E8,1.6224826253102347E8,2.5264070305020073E8],
"x":["CommonIntReverseBytesBenchmark","KarbideIntReverseBytesBenchmark","CommonShortReverseBytesBenchmark","KarbideShortReverseBytesBenchmark","CommonLongReverseBytesBenchmark","KarbideLongReverseBytesBenchmark"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"ymin"
},{
"type":"float",
"column":"ymax"
}]
}
}],
"spec_id":"451"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonIntReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 KarbideIntReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 CommonShortReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 KarbideShortReverseBytesBenchmark 
 
 
 
 
 
 
 
 
 CommonLongReverseBytesBenchma